##### ### The University of Melbourne, School of Computing and Information Systems

# COMP30027 Machine Learning, 2026 Semester 1

## Assignment 1: Income Classification with Naïve Bayes


**Student ID:** `1615557`


This iPython notebook is a template which you will use for your Assignment 1 submission.

**NOTE: YOU SHOULD ADD YOUR RESULTS, GRAPHS, AND FIGURES FROM YOUR OBSERVATIONS IN THIS FILE TO YOUR REPORT (the PDF file).** Results, figures, etc. which appear in this file but are NOT included in your report will not be marked.

**Adding proper comments to your code is MANDATORY. **


## 1. Supervised model training


In [4]:
import pandas as pd
import numpy as np
from collections import defaultdict as dd
from scipy.stats import norm

supervised_train_df = pd.read_csv("./datasets/adult_supervised_train.csv")
cleaned_df = supervised_train_df.dropna().copy()
cleaned_df["income"] = cleaned_df["income"].copy().map({"<=50K": 0, ">50K": 1})

classes = cleaned_df["income"].unique()
categorical_features = cleaned_df.select_dtypes(
    include=["object", "category"]
).columns.tolist()
continuous_features = cleaned_df.select_dtypes(include=[np.number]).columns.tolist()

# Group by income, sorting into classes 0 and 1, and then find the mean and variance of each of the columns where those
# features are continuous. grouped_stats is a MultiIndex
grouped_stats = cleaned_df.groupby("income")[continuous_features].agg(["mean", "var"])
# This moves the feature from part of the column to part of the row, meaning we can now index (class_c, feature_x)
# and get mean and variance for that combination
stacked_stats = grouped_stats.stack(level=0, future_stack=True)
# Convert the MultiIndex cleanly into a dictionary with keys (class_c, feature_x) and value being dictionaries
# themselves, whose keys are "mean" and "var". Overall, it gives the mean and variance for the rows in the dataframe
# with class = c and feature = x
stats_dict = stacked_stats.to_dict("index")

# this is the dictionary of distributions, ready to be used on new unseen test data points
distribs_dict = dd(float)
for key in stats_dict:
    mean = stats_dict[key]["mean"]
    std = np.sqrt(stats_dict[key]["var"])
    distribs_dict[key] = norm(loc=mean, scale=std)

# now must find P(x_j=v | c) for every combination of attribute, attribute value, and class
category_probabilities = dd(float)
# i guess i want the keys to be (class, attribute, attribute value) and the values to be the probabilities
ALPHA = 1
class_counts = {c: (cleaned_df["income"] == c).sum() for c in classes}
for cf in categorical_features:
    counts = pd.crosstab(
        cleaned_df["income"], cleaned_df[cf]
    )  # creates a 2d table of counts of how many times we have income = c and cf = v
    K_j = len(counts.columns)  # the number of distinct values for this feature
    for c in classes:
        for av in counts.columns:
            count = counts.loc[
                c, av
            ]  # means the row label is c and the column label is av. this gives the # of occurrences
            # where class is c and xj = av
            category_probabilities[(c, cf, av)] = (count + ALPHA) / (
                class_counts[c] + ALPHA * K_j
            )

R_vals = {}
for cf in categorical_features:
    for av in cleaned_df[cf].unique():
        p0 = category_probabilities[(0, cf, av)]  # chance of <=50K
        p1 = category_probabilities[(1, cf, av)]  # chance of >50K
        R_vals[(cf, av)] = p1 / p0

r_vals_list = [(key, val) for key, val in R_vals.items()]
r_vals_list.sort(
    key=lambda x: x[1], reverse=True
)  # this sort key sorts by the value in each tuple
print(r_vals_list)

[(('education', 'Prof-school'), np.float64(7.958687162957912)), (('education', 'Doctorate'), np.float64(7.135374697824336)), (('marital-status', 'Married-AF-spouse'), np.float64(4.2882067851373185)), (('education', 'Masters'), np.float64(3.5551339099645523)), (('native-country', 'Taiwan'), np.float64(3.383150965216618)), (('workclass', 'Self-emp-inc'), np.float64(3.250987093425404)), (('native-country', 'Hungary'), np.float64(3.0448358686949555)), (('occupation', 'Exec-managerial'), np.float64(2.768477763546732)), (('relationship', 'Wife'), np.float64(2.744950175060598)), (('relationship', 'Husband'), np.float64(2.5833814616287842)), (('marital-status', 'Married-civ-spouse'), np.float64(2.5514247558805545)), (('occupation', 'Prof-specialty'), np.float64(2.481622319759599)), (('native-country', 'India'), np.float64(2.3421814374576586)), (('native-country', 'Iran'), np.float64(2.2836269015212167)), (('education', 'Bachelors'), np.float64(2.2271039461887927)), (('native-country', 'Cambodi

## 2. Supervised model evaluation


In [ ]:
from sklearn.metrics import classification_report, accuracy_score
import math


test_set_df = pd.read_csv("./datasets/adult_test.csv")
test_set_df = test_set_df.dropna().copy()
test_set_df["income"] = test_set_df["income"].map({"<=50K": 0, ">50K": 1}).copy()


def calculate_likelihood_continuous(cj, xj):
    """
    Calculates the likelihood of an instance having attribute j being of value x, given that its class is c.
    """
    distrib = distribs_dict[
        cj
    ]  # get the normal distrib scaled for that class-feature combo with the mean and variance calculated from the data
    likelihood = distrib.pdf(xj)
    return max(
        likelihood, 1e-15
    )  # return tiny epsilon to prevent taking the log of 0 in case the probability is tiny


def calculate_posterior(likelihoods_list, c):
    """
    Calculates the posterior probability of an instance being of class c.
    """
    prior = class_counts[c] / len(cleaned_df)
    # start with the log prior and add on the log likelihoods
    log_posterior = math.log(prior)
    for l in likelihoods_list:
        log_posterior += math.log(l)
    # log is monotonic, so the highest log value will also be the highest non-log value
    return log_posterior


unseen_category_count = 0


def classify_instance(instance: pd.Series):
    """
    Returns the more likely class of a test instance, based on the values of its attributes.
    """
    global unseen_category_count
    leakproof_instance = instance.drop(labels=["income"], errors="ignore")
    # because probabilities are between 0 and 1, log(probability) will never be >0. The closer the probability
    # to 1, the less contribution adding it will make to the log sum, so the bigger (i.e. the less negative)
    # the sum of the logs is, the bigger the probability.
    max_log_posterior = -float("inf")
    best_class = None
    for c in classes:
        likelihoods_list = []
        for ft in leakproof_instance.index:
            av = leakproof_instance[ft]
            if ft in continuous_features:
                cj = (c, ft)  # class, feature
                likelihood = calculate_likelihood_continuous(cj=cj, xj=av)
                likelihoods_list.append(likelihood)
            else:
                likelihood = category_probabilities.get(
                    (c, ft, av), None
                )  # return tiny probability if there's a feature-value combo not found in training data
                if likelihood is None:
                    likelihood = 1e-7
                    if c == classes[0]:
                        unseen_category_count += 1
                likelihoods_list.append(likelihood)
        posterior = calculate_posterior(likelihoods_list, c)
        if posterior > max_log_posterior:
            max_log_posterior = posterior
            best_class = c
        if c == classes[1]: # if on second class
            
    posterior_ratio = 
    
    return best_class


test_set_df["prediction"] = test_set_df.apply(classify_instance, axis=1)
accuracy = accuracy_score(test_set_df["income"], test_set_df["prediction"])
print(
    "accuracy:",
    round(accuracy, 3),
)
print(classification_report(test_set_df["income"], test_set_df["prediction"]))
print("unseen_category_count:", unseen_category_count)

1
2
3
accuracy: 0.83
              precision    recall  f1-score   support

           0       0.85      0.93      0.89     11303
           1       0.72      0.52      0.61      3784

    accuracy                           0.83     15087
   macro avg       0.79      0.73      0.75     15087
weighted avg       0.82      0.83      0.82     15087

unseen_category_count: 1


## 3. Extending the model with semi-supervised training


## 4. Supervised model evaluation
